Imports e configurações

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

print("Bibliotecas carregadas!")

Bibliotecas carregadas!


Carregar colunas

In [2]:
colunas = [
    # localização
    "NO_REGIAO", "NO_UF", "SG_UF", "CO_UF",
    "NO_MUNICIPIO", "CO_MUNICIPIO",
    "TP_LOCALIZACAO",        # 1=Urbana, 2=Rural
    "TP_DEPENDENCIA",        # 1=Federal, 2=Estadual, 3=Municipal, 4=Privada

    # infraestrutura
    "IN_INTERNET",
    "IN_BIBLIOTECA",
    "IN_LABORATORIO_INFORMATICA",
    "IN_LABORATORIO_CIENCIAS",
    "IN_QUADRA_ESPORTES",
    "IN_BANHEIRO_PNE",
    "IN_ACESSIBILIDADE_INEXISTENTE",

    # matrículas por raça
    "QT_MAT_BAS",            # total de matrículas
    "QT_MAT_BAS_BRANCA",
    "QT_MAT_BAS_PRETA",
    "QT_MAT_BAS_PARDA",
    "QT_MAT_BAS_AMARELA",
    "QT_MAT_BAS_INDIGENA",

    # matrículas por sexo
    "QT_MAT_BAS_FEM",
    "QT_MAT_BAS_MASC",

    # docentes
    "QT_DOC_BAS",
]

df = pd.read_csv(
    "../data/raw/microdados_ed_basica_2023.csv",
    sep=";",
    encoding="latin-1",
    usecols=colunas,
    low_memory=False
)

print(f"Dados carregados: {df.shape[0]:,} escolas | {df.shape[1]} colunas")
df.head(3)

Dados carregados: 217,625 escolas | 24 colunas


,NO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,TP_DEPENDENCIA,TP_LOCALIZACAO,IN_BANHEIRO_PNE,IN_BIBLIOTECA,...,IN_INTERNET,QT_MAT_BAS,QT_MAT_BAS_FEM,QT_MAT_BAS_MASC,QT_MAT_BAS_BRANCA,QT_MAT_BAS_PRETA,QT_MAT_BAS_PARDA,QT_MAT_BAS_AMARELA,QT_MAT_BAS_INDIGENA,QT_DOC_BAS
0,Norte,Rondônia,RO,11,Porto Velho,1100205,2,1,1.0,1.0,...,1.0,69.0,24.0,45.0,18.0,0.0,32.0,0.0,0.0,16.0
1,Norte,Rondônia,RO,11,Porto Velho,1100205,3,1,0.0,0.0,...,1.0,225.0,103.0,122.0,65.0,6.0,147.0,0.0,1.0,10.0
2,Norte,Rondônia,RO,11,Porto Velho,1100205,4,1,1.0,1.0,...,1.0,1229.0,666.0,563.0,687.0,18.0,467.0,13.0,0.0,66.0


Traduzindo informações

In [3]:
df["REDE"] = df["TP_DEPENDENCIA"].map({
    1: "Federal", 2: "Estadual", 3: "Municipal", 4: "Privada"
})

df["LOCALIZACAO"] = df["TP_LOCALIZACAO"].map({
    1: "Urbana", 2: "Rural"
})

print("Categorias traduzidas!")
print(df[["REDE", "LOCALIZACAO"]].value_counts())

Categorias traduzidas!
REDE       LOCALIZACAO
Municipal  Rural          65436
           Urbana         65273
Privada    Urbana         51675
Estadual   Urbana         26586
           Rural           6960
Privada    Rural            970
Federal    Urbana           628
           Rural             97
Name: count, dtype: int64


Criar coluna de raça predominante

In [4]:
# converter colunas de raça para numérico
racas = ["QT_MAT_BAS_BRANCA", "QT_MAT_BAS_PRETA",
         "QT_MAT_BAS_PARDA", "QT_MAT_BAS_AMARELA", "QT_MAT_BAS_INDIGENA"]

df[racas] = df[racas].apply(pd.to_numeric, errors="coerce").fillna(0)
df["QT_MAT_BAS"] = pd.to_numeric(df["QT_MAT_BAS"], errors="coerce").fillna(0)

# raça com mais alunos em cada escola
nome_racas = {
    "QT_MAT_BAS_BRANCA":   "Branca",
    "QT_MAT_BAS_PRETA":    "Preta",
    "QT_MAT_BAS_PARDA":    "Parda",
    "QT_MAT_BAS_AMARELA":  "Amarela",
    "QT_MAT_BAS_INDIGENA": "Indígena"
}
df["RACA_PREDOMINANTE"] = df[racas].idxmax(axis=1).map(nome_racas)

# percentual de alunos pretos+pardos por escola
df["PERC_NEGROS"] = (
    (df["QT_MAT_BAS_PRETA"] + df["QT_MAT_BAS_PARDA"])
    / df["QT_MAT_BAS"].replace(0, np.nan)
) * 100

print("Colunas criadas!")
df[["RACA_PREDOMINANTE", "PERC_NEGROS"]].describe()

Colunas criadas!


,PERC_NEGROS
count,178476.000000
mean,40.025708
std,28.794739
min,0.000000
25%,14.432990
50%,35.766423
75%,63.905477
max,100.000000


In [5]:
df.to_parquet("../data/processed/escolas_limpo.parquet", index=False)
print(f"Salvo! {df.shape[0]:,} escolas prontas para análise.")

Salvo! 217,625 escolas prontas para análise.
